In [ ]:
import pandas as pd

# Full-set results already reported by the evaluation harness
results = pd.DataFrame([
    {"Task": "TruthfulQA MC1", "Baseline": 0.2938, "LoRA tuned": 0.2668},
    {"Task": "TruthfulQA MC2", "Baseline": 0.4587, "LoRA tuned": 0.4397},
])

results["Change (percentage points)"] = (
    results["LoRA tuned"] - results["Baseline"]
) * 100

results[["Baseline", "LoRA tuned"]] *= 100
results

In [ ]:
import json
from pathlib import Path
import numpy as np

BASE_DIR = Path(
    "results/truthfulqa_leng_20261027/baseline"
)

TUNED_DIR = Path(
    "results/truthfulqa_leng_20261027/tuned"
)

def load_task_scores(folder, task):
    files = [p for p in folder.rglob("*.jsonl") if task in p.name]
    if len(files) != 1:
        raise ValueError(
            f"Expected one {task} sample file in {folder}, found: {files}"
        )

    scores = {}
    with files[0].open() as f:
        for line in f:
            row = json.loads(line)
            score = row.get("acc")
            if score is None:
                score = row.get("metrics", {}).get("acc")
            if score is None:
                raise KeyError(f"No acc score found in a row of {files[0]}")
            scores[str(row["doc_id"])] = float(score)

    return scores

rng = np.random.default_rng(42)

for task in ["truthfulqa_mc1", "truthfulqa_mc2"]:
    baseline = load_task_scores(BASE_DIR, task)
    tuned = load_task_scores(TUNED_DIR, task)

    if baseline.keys() != tuned.keys():
        raise ValueError(f"Baseline and tuned question IDs differ for {task}")

    # Positive change means LoRA scored higher on the same questions.
    differences = np.array([tuned[k] - baseline[k] for k in baseline])

    bootstrap_means = np.array([
        rng.choice(differences, size=len(differences), replace=True).mean()
        for _ in range(10_000)
    ])

    low, high = np.quantile(bootstrap_means, [0.025, 0.975])
    print(
        f"{task}: change = {differences.mean() * 100:.2f} percentage points; "
        f"95% paired bootstrap interval = [{low * 100:.2f}, {high * 100:.2f}]"
    )